# Setup

In [ ]:
from collections.abc import Callable
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import numpy.typing as npt
import polars as pl
import seaborn as sns

from climate_attitudes.dataset import Dataset
from climate_attitudes.settings import Config
from climate_attitudes.visualisation import configure_mpl
from ising import Ising

configure_mpl(Path("../fonts/"))

np.set_printoptions(linewidth=200)

RANDOM_SEED = 202606081503

In [ ]:
config = Config(_env_file="../.env")
dataset = Dataset.load(
    config,
    name="reduced_no_imputation",
    with_imputation=False,
    verbose=False,
)
_, Y, _ = dataset.indices_to_numpy(kind="time-series", binarise=True, seed=RANDOM_SEED)

In [ ]:
model = Ising.fit(Y, update_method="sync", rng=RANDOM_SEED)
imodel = model.intervene(
    spins=np.array([2]), field_offset=np.array([0.5]), seed=RANDOM_SEED
)
model.reset(RANDOM_SEED)
imodel.reset(RANDOM_SEED)

# Shapley value calculation

We calculate shapley values for one individual ($i = 0$), for the task of predicting the next state of the spin at index 5. Our features are the initial spin states.

In [ ]:
y = Y[2]
y

To do so, we need to be able to calculate the expected value of this state, conditional on us only knowing the initial states of some spins, i.e., those in the set $Q$:

$$
\begin{align}
f_Q(\boldsymbol{s}) &= \mathbb{E}[f\mid S_i = s_i, \forall i \in Q] \\
&= \sum_{\boldsymbol{s}_{-Q}} P(\boldsymbol{s}_{-Q}) \cdot f(s')
\end{align}
$$

Where $s'$ comprises the values from $\boldsymbol{S}$ for those spins in $Q$, and $\boldsymbol{s}_{-Q}$ for all others.

While we don't know the values of the probabilities in the summation, we can approximate this value by drawing repeated samples based on the _previous_ timestep's state, and fixing the spins in $Q$ to their assigned values.

In [ ]:
def int_effect_1(model: Ising, imodel: Ising, Y: npt.NDArray[np.int64]) -> np.int64:
    effect = np.empty(Y.shape[0], dtype=np.float64)
    for i in range(Y.shape[0]):
        Y_null = Y[i]
        Y_int = Y[i]

        for _ in range(5):
            Y_null = model.step_parallel(Y_null)
            Y_int = imodel.step_parallel(Y_int)

        effect[i] = (Y_int - Y_null)[1]

    return effect


def s5(model: Ising, y: npt.NDArray[np.int64]) -> np.int64:
    y_next = model.step_parallel(y)
    return y_next[5]


def conditional_expectation[T: np.number](
    f: Callable[[Ising, npt.NDArray[np.int64]], T],
    Y: npt.NDArray[np.int64],
    Y_prev: npt.NDArray[np.int64],
    do: npt.NDArray[np.int64],
    model: Ising,
    imodel: Ising,
    samples: int = 100,
) -> np.float64:
    m, n = Y.shape
    inits = np.empty((samples, m, n), dtype=np.int64)
    for i in range(samples):
        for ind in range(m):
            inits[i, ind] = model.step_parallel(Y_prev[ind])
            imodel.step_parallel(Y_prev[ind])
            inits[i, ind, do] = Y[ind, do]

    evals = np.empty((samples, m), dtype=np.float64)
    for i in range(samples):
        evals[i] = f(model, imodel, inits[i])
    # for i, y_init in enumerate(inits):
    #     evals[i] = f(model, imodel, y_init)

    return evals.mean(axis=0)

In [ ]:
conditional_expectation(
    int_effect_1,
    Y[:10, 1],
    Y[:10, 0],
    do=np.array([0]),
    model=model,
    imodel=imodel,
    samples=10_000,
)

The next step is to define the contribution of a particular subset of features $Q$, which is:

$$\Delta_Q(\boldsymbol{s}) = f_Q(\boldsymbol{s}) - f_{\{\}}(\boldsymbol{s})$$

In [ ]:
def subset_contribution[T: np.number](
    f: Callable[[Ising, npt.NDArray[np.int64]], T],
    Y: npt.NDArray[np.int64],
    Y_prev: npt.NDArray[np.int64],
    do: npt.NDArray[np.int64],
    model: Ising,
    imodel: Ising,
    samples: int = 100,
) -> np.float64:
    f_null = conditional_expectation(
        f, Y, Y_prev, np.array([], dtype=np.int64), model, imodel, samples
    )
    f_q = conditional_expectation(f, Y, Y_prev, do, model, imodel, samples)
    return f_q - f_null

In [ ]:
subset_contribution(
    int_effect_1,
    Y[:3, 1],
    Y[:3, 0],
    do=np.array([0]),
    model=model,
    imodel=imodel,
    samples=10_000,
)

We now define the interactions:

$$\mathcal{I}_Q(\boldsymbol{s}) = \Delta_Q(\boldsymbol{s}) - \sum_{W \subset Q} \mathcal{I}_W(\boldsymbol{s})$$

We'll do this iteratively --- recursively is far too slow.

In [ ]:
def interaction[T: np.number](
    f: Callable[[Ising, npt.NDArray[np.int64]], T],
    y: npt.NDArray[np.int64],
    y_prev: npt.NDArray[np.int64],
    do: npt.NDArray[np.int64],
    model: Ising,
    imodel: Ising,
    samples: int = 100,
) -> np.float64:
    n = do.size
    g = np.empty(2**n, dtype=np.float64)
    for mask in range(1 << n):
        idx = np.fromiter((i for i in range(n) if mask & (1 << i)), dtype=int)
        g[mask] = subset_contribution(f, y, y_prev, do[idx], model, imodel, samples)

    evals = g.copy()

    # Möbius inversion on subsets
    for bit in range(n):
        for mask in range(1 << n):
            if mask & (1 << bit):
                evals[mask] -= evals[mask ^ (1 << bit)]

    return evals[-1]

In [ ]:
def compute_all_interactions[T: np.number](
    f: Callable[[Ising, npt.NDArray[np.int64]], T],
    Y: npt.NDArray[np.int64],
    Y_prev: npt.NDArray[np.int64],
    model: Ising,
    imodel: Ising,
    samples: int = 100,
):
    m, n = Y.shape
    spin_idxes = np.arange(n)
    g = np.empty((2**n, m), dtype=np.float64)
    for mask in range(1 << n):
        idx = np.fromiter((i for i in range(n) if mask & (1 << i)), dtype=int)
        g[mask] = subset_contribution(
            f, Y, Y_prev, spin_idxes[idx], model, imodel, samples
        )

    _interactions = g.copy()

    # Möbius inversion on subsets
    for bit in range(n):
        for mask in range(1 << n):
            if mask & (1 << bit):
                _interactions[mask] -= _interactions[mask ^ (1 << bit)]

    return _interactions

In [ ]:
f = compute_all_interactions(
    int_effect_1, Y[:3, 1], Y[:3, 0], model=model, imodel=imodel
)

Finally, we can compute the shapley value as the sum over all interactions not including the target spin, dividing by the number of features in each subset.

In [ ]:
def spin_shapley[T: np.number](
    f: Callable[[Ising, npt.NDArray[np.int64]], T],
    i: int,
    Y: npt.NDArray[np.int64],
    Y_prev: npt.NDArray[np.int64],
    model: Ising,
    imodel: Ising,
    samples: int = 100,
) -> np.float64:
    _interactions = compute_all_interactions(f, Y, Y_prev, model, imodel, samples)
    acc = np.zeros(Y.shape[0], dtype=np.float64)
    n = Y.shape[1]
    for mask in range(1 << n):
        if not mask:
            continue
        if not (mask & (1 << i)):
            acc += _interactions[mask] / (mask.bit_count() + 1)
    return acc


def shapley[T: np.number](
    f: Callable[[Ising, npt.NDArray[np.int64]], T],
    Y: npt.NDArray[np.int64],
    Y_prev: npt.NDArray[np.int64],
    model: Ising,
    samples: int = 100,
) -> npt.NDArray[np.float64]:
    evals = np.empty_like(Y, dtype=np.float64)
    for i in range(Y.shape[1]):
        evals[:, i] = spin_shapley(f, i, Y, Y_prev, model, imodel, samples)
    return evals

In [ ]:
m = 50
nsamples = 50

shap_vals = np.empty((m, 8), dtype=np.float64)

shap_vals = shapley(int_effect_1, Y[:m, 1], Y[:m, 0], model=model, samples=nsamples)

In [ ]:
cols = [
    "Belief CC",
    "CC Anthro",
    "CC Worry",
    "CC Worry (others)",
    "Weather worry",
    "Politics",
    "Climate impacts",
    "Climate policy",
]
plot_df = pl.DataFrame(
    {
        "Spin": [c for _ in range(m) for c in cols],
        "Shapley value": shap_vals.flatten(),
        # "State": shap_values.data.flatten(),
    }
)

In [ ]:
fig, ax = plt.subplots(constrained_layout=True)
sns.stripplot(plot_df, y="Spin", x="Shapley value", orient="h", s=3, ax=ax)

ax.set_yticks(np.arange(8), cols);